# Cost benchmark (paper Sect. 3.8, Table 6, Appendix A), one machine and one session

Every timing in the revised manuscript comes from this notebook. It uses the R_emb release checkpoint
(`checkpoints/remb_bimog_mcm_seed0.pt`) and the six benchmark organics of the original Part V benchmark
(12 000 binary evaluations). It reports medians of post-warm-up timings, and runs single-threaded on CPU, plus
the Apple-silicon GPU (MPS) if available. Run it on an otherwise idle machine.

In [1]:
import json, time, platform, numpy as np, torch, torch.nn as nn
from aiomfac_py import ActivityModel, Component
from aiomfac_py.s2as import smiles_to_components
import aiomfac_py.s2as._mapping as _m; _m.print = lambda *a, **k: None
import aiomfac_py.model as _model_mod
from aiomfac_py import lle
torch.set_num_threads(1)
print(platform.platform(), "|", platform.processor(), "| torch", torch.__version__, "| MPS:", torch.backends.mps.is_available())

meta = json.load(open("checkpoints/remb_bimog_mcm_meta.json")); VOCAB = meta["subgroup_vocabulary"]; mem = meta["members"][0]
class EmbedNet(nn.Module):
    def __init__(self, n_vocab, n_extra, emb=16):
        super().__init__(); self.n_vocab = n_vocab; self.emb = nn.Linear(n_vocab, emb, bias=False)
        self.head = nn.Sequential(nn.Linear(emb + n_extra, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(),
                                  nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2))
    def forward(self, x): return self.head(torch.cat([self.emb(x[:, :self.n_vocab]), x[:, self.n_vocab:]], 1))
net = EmbedNet(len(VOCAB), 2); net.load_state_dict(torch.load("checkpoints/" + mem["file"])); net.eval()
n_par = sum(p.numel() for p in net.parameters()); print("R_emb network parameters:", n_par)
em, es = np.array(mem["extra_mean"]), np.array(mem["extra_std"]); ym, ys = np.array(mem["y_mean"]), np.array(mem["y_std"])
W = {k: v.detach().numpy().astype(np.float32) for k, v in net.state_dict().items()}
def np_forward(x):
    h = np.concatenate([x[:, :len(VOCAB)] @ W["emb.weight"].T, x[:, len(VOCAB):]], 1)
    for i in (0, 2, 4):
        h = np.maximum(h @ W[f"head.{i}.weight"].T + W[f"head.{i}.bias"], 0)
    return h @ W["head.6.weight"].T + W["head.6.bias"]

SMILES = {"pinonic acid": "CC(=O)C1CC(CC(=O)O)C1(C)C", "pinic acid": "OC(=O)CC1CC(C(=O)O)C1(C)C", "glycerol": "OCC(O)CO",
          "acetic acid": "CC(=O)O", "sorbitol": "OCC(O)C(O)C(O)C(O)CO", "benzoic acid": "OC(=O)c1ccccc1"}
rng = np.random.RandomState(0); N_PER = 2000
def counts(comp):
    v = np.zeros(len(VOCAB), np.float32)
    for s, q in comp.subgroups: v[VOCAB.index(int(s))] = q
    return v
X_rows, cases = [], []
for nm, smi in SMILES.items():
    comps = smiles_to_components([smi]).components
    w = rng.uniform(0.02, 0.98, N_PER); T = rng.uniform(273.15, 313.15, N_PER)
    cases.append((nm, comps, w, T))
    X_rows.append(np.column_stack([np.tile(counts(comps[1]), (N_PER, 1)), (np.column_stack([w, (T - 293.15) / 20]) - em) / es]))
X = np.concatenate(X_rows).astype(np.float32); print("benchmark inputs:", X.shape)
assert np.max(np.abs(np_forward(X[:500]) - net(torch.tensor(X[:500])).detach().numpy())) < 1e-5
print("max |NumPy - PyTorch| on 12 000 points:", float(np.max(np.abs(np_forward(X) - net(torch.tensor(X)).detach().numpy()))))

macOS-26.6.2-arm64-arm-64bit-Mach-O | arm | torch 2.14.1 | MPS: True
R_emb network parameters: 36594
benchmark inputs: (12000, 57)
max |NumPy - PyTorch| on 12 000 points: 9.5367431640625e-07


In [2]:
def per_call(fn, args_list, warm=20):
    for a in args_list[:warm]: fn(a)
    t = np.empty(len(args_list))
    for i, a in enumerate(args_list):
        t0 = time.perf_counter(); fn(a); t[i] = time.perf_counter() - t0
    return float(np.median(t))
def batched(fn, x, reps=20, warm=3):
    for _ in range(warm): fn(x)
    t = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(x); t.append(time.perf_counter() - t0)
    return float(np.median(t)) / len(x)

res = {}
# AIOMFAC direct: model built once per organic, one evaluate() per point
t_a = []
for nm, comps, w, T in cases:
    m = ActivityModel(comps)
    t_a.append(per_call(lambda a: m.evaluate([a[0], 1 - a[0]], a[1], basis="mass"), list(zip(w, T))))
res["AIOMFAC, direct call"] = float(np.median(t_a))
xt = torch.tensor(X)
with torch.no_grad():
    res["Surrogate, PyTorch, one at a time"] = per_call(lambda i: net(xt[i:i + 1]), list(range(len(X))))
    res["Surrogate, PyTorch, batched (12 000)"] = batched(lambda x: net(x), xt)
res["Surrogate, NumPy, one at a time"] = per_call(lambda i: np_forward(X[i:i + 1]), list(range(len(X))))
res["Surrogate, NumPy, batched (12 000)"] = batched(np_forward, X)
if torch.backends.mps.is_available():
    dev = torch.device("mps"); net_g = EmbedNet(len(VOCAB), 2).to(dev); net_g.load_state_dict(net.state_dict()); net_g.eval()
    def g1(i):
        with torch.no_grad(): out = net_g(torch.tensor(X[i:i + 1], device=dev)).cpu()
        return out
    def gb(x):
        with torch.no_grad(): out = net_g(torch.tensor(x, device=dev)); torch.mps.synchronize(); return out.cpu()
    res["Surrogate, Apple GPU (MPS), one at a time"] = per_call(g1, list(range(2000)))
    res["Surrogate, Apple GPU (MPS), batched (12 000)"] = batched(gb, X)
base = res["AIOMFAC, direct call"]
print(f"{'method':48s} {'µs/eval':>9s} {'speed-up':>9s}")
for k, v in res.items(): print(f"{k:48s} {1e6*v:9.2f} {base/v:9.1f}")

method                                             µs/eval  speed-up
AIOMFAC, direct call                                 75.02       1.0
Surrogate, PyTorch, one at a time                    36.92       2.0
Surrogate, PyTorch, batched (12 000)                  0.45     166.7
Surrogate, NumPy, one at a time                      14.13       5.3
Surrogate, NumPy, batched (12 000)                    0.82      91.5
Surrogate, Apple GPU (MPS), one at a time          1152.94       0.1
Surrogate, Apple GPU (MPS), batched (12 000)          0.52     143.7


In [3]:
# per-molecule fixed costs and end-to-end cost per molecule (Table 6)
def med(fn, n):
    fn(); t = []
    for _ in range(n):
        t0 = time.perf_counter(); fn(); t.append(time.perf_counter() - t0)
    return float(np.median(t))
s2as_t, build_t = [], []
for nm, smi in SMILES.items():
    s2as_t.append(med(lambda: smiles_to_components([smi]), 30))
    comps = smiles_to_components([smi]).components
    build_t.append(med(lambda: ActivityModel(comps), 100))
print(f"S2AS decomposition: {1e3*min(s2as_t):.2f}–{1e3*max(s2as_t):.2f} ms (median {1e3*np.median(s2as_t):.2f}); "
      f"model construction {1e3*np.median(build_t):.3f} ms")
t_s2as, t_build, t_eval, t_sur = np.median(s2as_t), np.median(build_t), res["AIOMFAC, direct call"], res["Surrogate, NumPy, batched (12 000)"]
rows = []
for P in (1, 10, 100, 1000, 10000):
    a = t_s2as + t_build + P * t_eval; s = t_s2as + P * t_sur
    rows.append((P, 1e3 * a, 1e3 * s, a / s))
print(f"{'points/molecule':>15s} {'AIOMFAC ms':>11s} {'surrogate ms':>13s} {'speed-up':>9s}")
for r in rows: print(f"{r[0]:15d} {r[1]:11.2f} {r[2]:13.2f} {r[3]:9.1f}")
json.dump({"per_eval_s": res, "s2as_s": s2as_t, "build_s": build_t, "end_to_end": rows, "n_params": n_par},
          open("cost_benchmark_remb.json", "w"), indent=1)

S2AS decomposition: 0.66–2.44 ms (median 1.79); model construction 0.043 ms
points/molecule  AIOMFAC ms  surrogate ms  speed-up
              1        1.91          1.79       1.1
             10        2.58          1.80       1.4
            100        9.34          1.87       5.0
           1000       76.86          2.61      29.4
          10000      752.04          9.99      75.3


In [4]:
# multicomponent scaling and phase-equilibrium call count (same session)
import pandas as pd
mcm = pd.read_csv("mcm_v331_species.tsv", sep="\t", comment="*")
from rdkit import Chem, RDLogger; RDLogger.DisableLog("rdApp.*")
pool = []
for s in mcm.Smiles.astype(str):
    mol = Chem.MolFromSmiles(s)
    if mol is None or any(a.GetNumRadicalElectrons() for a in mol.GetAtoms()) or not {a.GetSymbol() for a in mol.GetAtoms()} <= {"C", "H", "O", "N"} or mol.GetNumHeavyAtoms() < 2:
        continue
    r = smiles_to_components([s])
    if not r.removed and r.n_unmatched_atoms[0] == 0: pool.append(r.components[1].subgroups)
    if len(pool) >= 400: break
rng = np.random.RandomState(1); WATER = Component(1, "Water", ((16, 1),))
for n_org in (1, 2, 5, 10, 20, 40):
    picks = [pool[i] for i in rng.choice(len(pool), n_org, replace=False)]
    m = ActivityModel([WATER] + [Component(k + 2, f"o{k}", tuple(p)) for k, p in enumerate(picks)])
    x = np.full(n_org + 1, 1.0 / (n_org + 1))
    print(f"{n_org:3d} organics: {1e6*med(lambda: m.evaluate(x, 298.15, basis='mass'), 200):7.1f} µs per evaluation")
calls = {"n": 0}; orig = _model_mod.sr_terms
def counting(*a, **k):
    calls["n"] += 1; return orig(*a, **k)
_model_mod.sr_terms = counting
try:
    t0 = time.perf_counter(); lle.solve_pep(smiles_to_components(["CCCCO"]).components, np.array([0.5, 0.5]), 298.15)
    print(f"1-butanol/water LLE: {calls['n']} short-range evaluations, {1e3*(time.perf_counter()-t0):.0f} ms")
finally:
    _model_mod.sr_terms = orig

  1 organics:    74.2 µs per evaluation
  2 organics:    89.7 µs per evaluation
  5 organics:   133.8 µs per evaluation
 10 organics:   194.3 µs per evaluation
 20 organics:   332.8 µs per evaluation


 40 organics:   638.0 µs per evaluation
1-butanol/water LLE: 146 short-range evaluations, 15 ms
